In [0]:
import dlt
from pyspark.sql.functions import col, lower, to_timestamp

# Create target streaming table for merge
dlt.create_streaming_table(
    name="electroniz_silver_product_reviews"
)

# Temporary view with transformations and referential integrity check
@dlt.view
def product_reviews_clean():
    # Read streaming source - bronze product reviews
    reviews = spark.readStream.table("electroniz_catalog.electroniz_bronze_schema.electroniz_bronze_product_reviews")

    # Step 1 & 2: Parse map data and extract/rename fields with type casting
    parsed = reviews.select(
        col("data")["product/productId"].cast("string").alias("product_Id"),
        col("data")["product_name"].cast("string").alias("product_name"),
        col("data")["review/userId"].cast("string").alias("userId"),
        col("data")["review/profileName"].cast("string").alias("profileName"),
        col("data")["review/helpfulness"].cast("string").alias("helpfulness"),
        col("data")["review/score"].cast("double").alias("score"),
        to_timestamp(col("data")["review/time"].cast("bigint")).alias("review_time"),
        col("data")["review/summary"].cast("string").alias("summary"),
        col("data")["review/text"].cast("string").alias("review_text"),
        col("_ingested_at").alias("ingested_at")
    )

    # Step 3: Read products dimension table (static read for referential integrity)
    products = spark.read.table("electroniz_catalog.electroniz_bronze_schema.electroniz_bronze_products")

    # Data Quality: INNER JOIN with products on case-insensitive product_name match
    # This automatically drops reviews where product_name doesn't match any product
    result = parsed.join(
        products,
        lower(parsed["product_name"]) == lower(products["product_name"]),
        "inner"
    ).select(
        parsed["product_Id"],
        parsed["product_name"],
        parsed["userId"],
        parsed["profileName"],
        parsed["helpfulness"],
        parsed["score"],
        parsed["review_time"],
        parsed["summary"],
        parsed["review_text"],
        parsed["ingested_at"]
    )

    return result

# Apply changes: Merge into target table using composite key (product_Id + userId)
dlt.apply_changes(
    target="electroniz_silver_product_reviews",
    source="product_reviews_clean",
    keys=["product_Id", "userId"],
    sequence_by=col("ingested_at")
)